In [ ]:
#| default_exp gen

In [1]:
#| hide
import nbdev; nbdev.nbdev_export()

In [14]:
#| export
def bad_points(tokenizer, point):
    result = []
    for i in range(2, 50):
        code = tokenizer.encode(point*i)
        if len(code) == 1:
            result += [code]
    return result

def bad_words(tokenizer, allow_linebreak):
    bad_symbols = ['[','(','\xa0','*','­', '~', '_', '\\', '\n\n', '\uf04a', '\ufeff', '\u2028']
    bad_words_ids = [tokenizer.encode(s) for s in bad_symbols]
    
    eot = tokenizer.encode('a<|endoftext|>')[1]
    if eot: bad_words_ids += [[eot]]
    
    for point in ['.','*','_','-','\xa0','!']:
        bad_words_ids += bad_points(tokenizer, point)
    linebreaks = [tokenizer.encode(s) for s in ['\n', ' \n']]
    bad_words_ids += [] if allow_linebreak else linebreaks
    return bad_words_ids

In [15]:
#| export
from front.common import process_seq

def generate(model, tokenizer, seq_length, prompt, length:int, num_samples:int, allow_linebreak:bool):
    encoded_prompt = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").cuda()
    encoded_prompt = encoded_prompt[:,length-(seq_length-1):]
    output_sequences = model.generate(
            input_ids=encoded_prompt,
            max_length= length + len(encoded_prompt[0]),
            temperature=1,
            top_k=0,
            top_p=0.9,
            do_sample=True,
            num_return_sequences=num_samples,
            no_repeat_ngram_size=2,
            #repetition_penalty=5.0,
            #typical_p=0.9,
            bad_words_ids = bad_words(tokenizer, allow_linebreak),
            early_stopping=True,
            num_beams=1
        )
    if len(output_sequences.shape) > 2:
            output_sequences.squeeze_()
    generated_sequences = []
    for generated_sequence_idx, generated_sequence in enumerate(output_sequences):
        generated_sequence = generated_sequence.tolist()
        text = tokenizer.decode(generated_sequence, clean_up_tokenization_spaces=True)
        total_sequence = text[len(tokenizer.decode(encoded_prompt[0], clean_up_tokenization_spaces=True)) :]
        generated_sequences.append(total_sequence)

    return process_seq(generated_sequences)